# LangChain 综合复现 Notebook（阿里云百炼版）

## 使用建议

1. 优先先运行前面的“00 环境检查与统一配置”。
2. 建议按顺序逐段执行；如果你已经熟悉某一节，也可以直接跳转。
3. 本 Notebook 统一使用阿里云百炼：
   - 优先读取 `DASHSCOPE_API_KEY`
   - 如果当前进程拿不到，会自动回退读取 Windows 用户级环境变量
   - 同时兼容 `ALIYUN_BAILIAN_API_KEY`
4. 对本地模型、OpenAI 兼容接口、部分高阶检索章节，Notebook 会先检查条件，不满足时给出跳过原因。

## 推荐安装命令

```bash
python -m pip install -U langchain langchain-community langchain-openai langchain-text-splitters chromadb pypdf dashscope beautifulsoup4 lark defusedxml
```


## 00. 环境检查与统一配置

### 00.1 这一段做什么

- 统一定位当前 Notebook 目录与素材目录
- 自动读取 Windows 用户级环境变量中的百炼 API Key
- 提供后续所有章节都会复用的辅助函数
- 如果某一节依赖不满足，会打印“跳过原因”而不是直接中断


In [ ]:
from __future__ import annotations

import importlib.util
import os
import warnings
from pathlib import Path

try:
    import winreg
except ImportError:
    winreg = None

try:
    from langchain_core._api.deprecation import LangChainDeprecationWarning
    warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)
except Exception:
    pass

warnings.filterwarnings("ignore", message="The class `Chroma` was deprecated.*")


def locate_project_root() -> Path:
    signatures = [("txt/1.md", "txt/test.html", "pdf/2312.04005.pdf")]
    seen: set[Path] = set()
    candidates: list[Path] = []

    def add_candidate(path: Path) -> None:
        resolved = path.resolve()
        if resolved not in seen and resolved.exists() and resolved.is_dir():
            seen.add(resolved)
            candidates.append(resolved)

    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        add_candidate(base)
        try:
            child_dirs = [child for child in base.iterdir() if child.is_dir()]
        except PermissionError:
            child_dirs = []
        for child in child_dirs:
            add_candidate(child)
            try:
                grandchild_dirs = [grandchild for grandchild in child.iterdir() if grandchild.is_dir()]
            except PermissionError:
                grandchild_dirs = []
            for grandchild in grandchild_dirs:
                add_candidate(grandchild)

    for candidate in candidates:
        if all((candidate / rel).exists() for rel in signatures[0]):
            return candidate

    raise FileNotFoundError("没有找到 02_bilibili_langchain_bailian 目录，请确认从仓库目录中打开 Notebook。")


PROJECT_DIR = locate_project_root()
os.chdir(PROJECT_DIR)
ROOT_DIR = PROJECT_DIR
TEXT_DIR = PROJECT_DIR / "txt"
PDF_DIR = PROJECT_DIR / "pdf"
HTML_DIR = PROJECT_DIR / "html"
CACHE_DIR = PROJECT_DIR / "_combined_cache"
CHROMA_DIR = PROJECT_DIR / "_combined_chroma"
CACHE_DIR.mkdir(exist_ok=True)
CHROMA_DIR.mkdir(exist_ok=True)


def has_module(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None


def read_windows_user_env(name: str) -> str | None:
    if winreg is None:
        return None
    try:
        with winreg.OpenKey(winreg.HKEY_CURRENT_USER, r"Environment") as key:
            value, _ = winreg.QueryValueEx(key, name)
            return value
    except FileNotFoundError:
        return None


def resolve_dashscope_api_key() -> str | None:
    return (
        os.getenv("DASHSCOPE_API_KEY")
        or os.getenv("ALIYUN_BAILIAN_API_KEY")
        or read_windows_user_env("DASHSCOPE_API_KEY")
        or read_windows_user_env("ALIYUN_BAILIAN_API_KEY")
    )


DASHSCOPE_API_KEY = resolve_dashscope_api_key()
if DASHSCOPE_API_KEY:
    os.environ["DASHSCOPE_API_KEY"] = DASHSCOPE_API_KEY

HAS_DASHSCOPE = bool(DASHSCOPE_API_KEY)
HAS_LANGCHAIN_OPENAI = has_module("langchain_openai")
HAS_LARK = has_module("lark")
HAS_BS4 = has_module("bs4")
HAS_TORCH = has_module("torch")
HAS_TRANSFORMERS = has_module("transformers")
OPENAI_COMPAT_BASE_URL = os.getenv("OPENAI_COMPAT_BASE_URL") or "https://dashscope.aliyuncs.com/compatible-mode/v1"


def section_ready(section_id: str, condition: bool, reason: str) -> bool:
    if condition:
        print(f"[{section_id}] ready")
        return True
    print(f"[{section_id}] skipped: {reason}")
    return False


def short_text(text: str, limit: int = 180) -> str:
    text = text.strip().replace("\n", " ")
    return text if len(text) <= limit else text[:limit] + " ..."


def make_chat_model(model_name: str = "qwen-plus", temperature: float = 0.2):
    from langchain_community.chat_models import ChatTongyi
    return ChatTongyi(model=model_name, temperature=temperature)


def make_llm(model_name: str = "qwen-plus", temperature: float = 0.2):
    from langchain_community.llms import Tongyi
    return Tongyi(model=model_name, temperature=temperature)


print(f"PROJECT_DIR = {PROJECT_DIR}")
print(f"TEXT_DIR exists = {TEXT_DIR.exists()}")
print(f"PDF_DIR exists = {PDF_DIR.exists()}")
print(f"HTML_DIR exists = {HTML_DIR.exists()}")
print(f"DASHSCOPE_API_KEY = {'SET' if HAS_DASHSCOPE else 'NOT SET'}")
print(f"langchain_openai = {HAS_LANGCHAIN_OPENAI}")
print(f"lark = {HAS_LARK}")
print(f"beautifulsoup4 = {HAS_BS4}")
print(f"torch = {HAS_TORCH}")
print(f"transformers = {HAS_TRANSFORMERS}")


## 01-Langchain

### 01.1 基础链：Prompt + Model + OutputParser

这一节对应最基础的 LangChain 组合方式。我们用一个简单链条把“提示模板 -> 百炼聊天模型 -> 字符串解析器”串起来。


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

if section_ready("01.1", HAS_DASHSCOPE, "未找到 DASHSCOPE_API_KEY / ALIYUN_BAILIAN_API_KEY"):
    prompt = ChatPromptTemplate.from_template(
        "请围绕“{topic}”写一段不超过 80 字的中文学习提示，不要使用表情符号。"
    )
    chain = prompt | make_chat_model(temperature=0.1) | StrOutputParser()
    result = chain.invoke({"topic": "LangChain 入门"})
    print(result)


## 02-ChatGLM4接入Langchain

### 02.1 章节改造说明

原笔记本这一节主要演示“把第三方聊天模型接入 LangChain”。这里统一改为阿里云百炼聊天模型，保留“聊天消息列表”的学习重点。


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

if section_ready("02.1", HAS_DASHSCOPE, "未找到百炼 API Key"):
    chat_model = make_chat_model(model_name="qwen-plus", temperature=0)
    messages = [
        SystemMessage(content="你是一名 LangChain 中文助教，回答简洁、纯文本。"),
        HumanMessage(content="请用两句话解释 LangChain 的主要作用。"),
    ]
    response = chat_model.invoke(messages)
    print(response.content)


### 02.2 流式输出体验

这一节对照原始示例中的流式调用，帮助你理解 `stream()` 在 LangChain 里的使用方式。


In [ ]:
if section_ready("02.2", HAS_DASHSCOPE, "未找到百炼 API Key"):
    chat_model = make_chat_model(model_name="qwen-plus", temperature=0)
    print("stream result:")
    for chunk in chat_model.stream("请用一句话解释什么是 RAG，不要使用表情符号。"):
        content = getattr(chunk, "content", "")
        if content:
            print(content, end="")
    print()


## 03-langchain使用本地大模型

### 03.1 自定义 LLM 包装器的最小示例

原笔记本这里是“如何把本地模型包成 LangChain 可调用对象”。为了保证 Notebook 默认可跑，这里先用一个轻量的 `DemoLocalLLM` 演示接口形状。


In [ ]:
from langchain_core.language_models.llms import LLM

class DemoLocalLLM(LLM):
    @property
    def _llm_type(self) -> str:
        return "demo-local-llm"

    def _call(self, prompt: str, stop=None, run_manager=None, **kwargs) -> str:
        return f"[本地模型演示输出] 收到提示词：{prompt[:60]}"

demo_llm = DemoLocalLLM()
print(demo_llm.invoke("你好，请演示一个本地 LLM 包装器会如何工作。"))


## 04-本地部署开源大模型类OPENAI接口接入Langchain

### 04.1 OpenAI 兼容模式接入

这一节保留“OpenAI-compatible 接口”的学习目标。这里直接演示用 `ChatOpenAI` 接入阿里云百炼兼容端点。


In [ ]:
if section_ready("04.1", HAS_DASHSCOPE and HAS_LANGCHAIN_OPENAI, "缺少百炼 API Key 或 langchain-openai"):
    from langchain_openai import ChatOpenAI

    compat_model = ChatOpenAI(
        api_key=DASHSCOPE_API_KEY,
        base_url=OPENAI_COMPAT_BASE_URL,
        model="qwen-plus",
        temperature=0,
    )
    response = compat_model.invoke("请只回复：OpenAI 兼容模式可用")
    print(response.content)


## 05-Langchain提示词模块

### 05.1 PromptTemplate 基础

先看最经典的字符串模板写法。


In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    "请用{style}风格，写一句关于“{topic}”的学习提醒。"
)
print(prompt_template.format(style="清爽", topic="LangChain"))


### 05.2 ChatPromptTemplate 与链式调用

这一段对应原笔记本里最常见的 LCEL 组合方式。


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

if section_ready("05.2", HAS_DASHSCOPE, "未找到百炼 API Key"):
    chat_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "你是一名中文学习助理，回答纯文本。"),
            ("human", "请根据主题“{topic}”输出一条学习建议。"),
        ]
    )
    chain = chat_prompt | make_chat_model(temperature=0.1) | StrOutputParser()
    print(chain.invoke({"topic": "提示词模板"}))


## 06-提示词案例选择器

### 06.1 按长度选择 Few-shot 示例

这一节先保留最稳的 `LengthBasedExampleSelector`，方便理解“示例越多越长，提示词越大”的取舍。


In [ ]:
from langchain_core.example_selectors import LengthBasedExampleSelector
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

examples = [
    {"input": "我学不动了", "output": "先把任务拆成 5 分钟小动作。"},
    {"input": "我有点乱", "output": "先列出目标、输入和输出。"},
    {"input": "我想学 RAG", "output": "先从最小检索链开始。"},
]
example_prompt = PromptTemplate.from_template("用户：{input}\n助手：{output}")
selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=90,
)
few_shot_prompt = FewShotPromptTemplate(
    example_selector=selector,
    example_prompt=example_prompt,
    prefix="请参考下面示例回答。",
    suffix="用户：{query}\n助手：",
    input_variables=["query"],
)
print(few_shot_prompt.format(query="我准备复现 LangChain，但是不知道先做什么"))


## 07-模型 (LLMs) 的类似少样本提示

### 07.1 语义相似示例选择器

这里对应“根据语义相似度自动选示例”。为了避免依赖本地向量模型，直接改为百炼嵌入 + Chroma。


In [ ]:
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_community.vectorstores import Chroma

if section_ready("07.1", HAS_DASHSCOPE, "未找到百炼 API Key"):
    examples = [
        {"input": "我很累", "output": "先休息十分钟，再回来继续。"},
        {"input": "我想学习 RAG", "output": "先搭最小可运行检索链。"},
        {"input": "我想拖延", "output": "把目标拆成最小动作。"},
    ]
    example_prompt = PromptTemplate.from_template("用户：{input}\n助手：{output}")
    selector = SemanticSimilarityExampleSelector.from_examples(
        examples,
        make_embeddings(),
        Chroma,
        k=2,
        collection_name="section_07_selector",
        persist_directory=str(CHROMA_DIR / "section_07_selector"),
    )
    prompt = FewShotPromptTemplate(
        example_selector=selector,
        example_prompt=example_prompt,
        prefix="请参考最相近的示例回答。",
        suffix="用户：{query}\n助手：",
        input_variables=["query"],
    )
    print(prompt.format(query="我最近学 LangChain 学得有点乱"))


## 08-对话提示词工程

### 08.1 FewShotChatMessagePromptTemplate

这一节保留“聊天消息风格”的 few-shot 模板思路。


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate

examples = [
    {"input": "你好", "output": "你好呀，我来陪你梳理思路。"},
    {"input": "我有点乱", "output": "没关系，我们先拆成最小一步。"},
]
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一位耐心的中文学习助理。"),
        few_shot,
        ("human", "{question}"),
    ]
)
messages = final_prompt.format_messages(question="我想学 LangChain，从哪里开始？")
for msg in messages:
    print(type(msg).__name__, "=>", short_text(msg.content))


## 09-提示词里模板中的模板

### 09.1 MessagesPlaceholder 与 partial

这一节展示“模板里再放消息占位符”的组合方式。


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一名中文老师。"),
        MessagesPlaceholder("history"),
        ("human", "{question}"),
    ]
).partial(question="请总结上一轮对话重点")

messages = prompt.format_messages(
    history=[
        ("human", "你好"),
        ("ai", "你好，我可以帮你整理笔记。"),
    ]
)

for msg in messages:
    print(type(msg).__name__, "=>", msg.content)


## 10-管道提示词

### 10.1 用可复用子模板拼接最终提示

新版本里 `PipelinePromptTemplate` 的位置已经变化较大，这里用“子模板先格式化，再注入最终模板”的方式保留原学习目标。


In [ ]:
from langchain_core.prompts import PromptTemplate

intro_template = PromptTemplate.from_template("你是一名{role}。")
task_template = PromptTemplate.from_template("现在请完成任务：{task}")
final_template = PromptTemplate.from_template("{intro}\n{task_block}")

intro = intro_template.format(role="LangChain 助教")
task_block = task_template.format(task="用一句话解释什么是 LCEL")
combined_prompt = final_template.format(intro=intro, task_block=task_block)
print(combined_prompt)


## 11-缓存

### 11.1 InMemoryCache

这一节演示 LangChain 的缓存层如何减少重复请求。


In [ ]:
import time

from langchain_community.cache import InMemoryCache
from langchain_core.globals import set_llm_cache

if section_ready("11.1", HAS_DASHSCOPE, "未找到百炼 API Key"):
    set_llm_cache(InMemoryCache())
    model = make_chat_model(model_name="qwen-plus", temperature=0)
    prompt = "请只回复：缓存测试"

    start = time.perf_counter()
    first = model.invoke(prompt)
    first_cost = time.perf_counter() - start

    start = time.perf_counter()
    second = model.invoke(prompt)
    second_cost = time.perf_counter() - start

    print("first =>", first.content, "| elapsed =", round(first_cost, 3), "s")
    print("second =>", second.content, "| elapsed =", round(second_cost, 3), "s")


### 11.2 SQLiteCache

相比内存缓存，SQLite 缓存更适合跨 Notebook / 跨进程复用。


In [ ]:
from langchain_community.cache import SQLiteCache
from langchain_core.globals import set_llm_cache

sqlite_cache_path = CACHE_DIR / "langchain_cache.sqlite"
set_llm_cache(SQLiteCache(database_path=str(sqlite_cache_path)))
print("SQLite cache path =", sqlite_cache_path)
print("cache file exists =", sqlite_cache_path.exists())


## 12-CSV_时间_枚举解析器

### 12.1 CSV / 时间 / 枚举解析器

这一节把三个常见解析器放在一起演示。


In [ ]:
from enum import Enum

from langchain_classic.output_parsers import DatetimeOutputParser, EnumOutputParser
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate

if section_ready("12.1", HAS_DASHSCOPE, "未找到百炼 API Key"):
    model = make_chat_model(model_name="qwen-plus", temperature=0)

    list_parser = CommaSeparatedListOutputParser()
    list_prompt = PromptTemplate(
        template="请给我三个和{topic}有关的关键词。\n{format_instructions}",
        input_variables=["topic"],
        partial_variables={"format_instructions": list_parser.get_format_instructions()},
    )
    list_result = model.invoke(list_prompt.format(topic="RAG")).content
    print("CSV =>", list_parser.parse(list_result))

    class Difficulty(str, Enum):
        beginner = "beginner"
        intermediate = "intermediate"
        advanced = "advanced"

    enum_parser = EnumOutputParser(enum=Difficulty)
    enum_prompt = PromptTemplate(
        template="请判断“{topic}”更适合哪种学习难度，只输出一个枚举值。\n{format_instructions}",
        input_variables=["topic"],
        partial_variables={"format_instructions": enum_parser.get_format_instructions()},
    )
    enum_result = model.invoke(enum_prompt.format(topic="LangChain 基础教程")).content
    print("ENUM =>", enum_parser.parse(enum_result))

    dt_parser = DatetimeOutputParser()
    dt_prompt = PromptTemplate(
        template="请把 2026-04-15 14:30 转成标准 datetime 字符串。\n{format_instructions}",
        input_variables=[],
        partial_variables={"format_instructions": dt_parser.get_format_instructions()},
    )
    dt_result = model.invoke(dt_prompt.format()).content
    print("DATETIME =>", dt_parser.parse(dt_result))


## 13-JSON parser

### 13.1 JSON 输出解析

这一节保留“让模型按指定结构输出 JSON”的核心思路。


In [ ]:
from pydantic import BaseModel, Field

from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

class StudyPlan(BaseModel):
    topic: str = Field(description="学习主题")
    steps: list[str] = Field(description="学习步骤")

if section_ready("13.1", HAS_DASHSCOPE, "未找到百炼 API Key"):
    parser = JsonOutputParser(pydantic_object=StudyPlan)
    prompt = PromptTemplate(
        template="请为 {topic} 生成一个 3 步学习计划。\n{format_instructions}",
        input_variables=["topic"],
        partial_variables={"format_instructions": parser.get_format_instructions()},
    )
    chain = prompt | make_chat_model(model_name="qwen-plus", temperature=0) | parser
    print(chain.invoke({"topic": "LangChain RAG"}))


## 14-重试解析器 / 14-xml解析器

### 14.1 XML 解析与修复输出

这里把原来的两个 14 系列笔记本合并展示：先看 XML，再看错误输出修复。


In [ ]:
from pydantic import BaseModel, Field

from langchain_classic.output_parsers.fix import OutputFixingParser
from langchain_core.output_parsers import PydanticOutputParser, XMLOutputParser
from langchain_core.prompts import PromptTemplate

if section_ready("14.1", HAS_DASHSCOPE, "未找到百炼 API Key"):
    model = make_chat_model(model_name="qwen-plus", temperature=0)

    xml_parser = XMLOutputParser(tags=["topic", "summary"])
    xml_prompt = PromptTemplate(
        template="请输出一个很短的 XML，总结 {topic}。\n{format_instructions}",
        input_variables=["topic"],
        partial_variables={"format_instructions": xml_parser.get_format_instructions()},
    )
    xml_text = model.invoke(xml_prompt.format(topic="LangChain")).content
    print("XML =>", xml_parser.parse(xml_text))

    class Outline(BaseModel):
        title: str = Field(description="标题")
        bullets: list[str] = Field(description="要点列表")

    parser = PydanticOutputParser(pydantic_object=Outline)
    fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=model)
    broken_output = '{"title": "LangChain", "bullets": "提示词, 检索, 代理"}'
    print("FIXED =>", fixing_parser.parse(broken_output))


## 15-自定义大模型输出解析器

### 15.1 用 RunnableLambda 做自定义解析

新版本 LangChain 更推荐用 Runnable 组合自定义解析逻辑。


In [ ]:
from langchain_core.runnables import RunnableLambda

def parse_outline(text: str) -> dict:
    lines = [line.strip("- ").strip() for line in text.splitlines() if line.strip()]
    return {
        "title": lines[0] if lines else "",
        "bullets": lines[1:] if len(lines) > 1 else [],
    }

parser = RunnableLambda(parse_outline)
sample_text = "LangChain 学习地图\n- 基础调用\n- Prompt 模板\n- 检索增强生成"
print(parser.invoke(sample_text))


## 16-数据检索

### 16.1 最小可运行 RAG 检索链

这一节把“向量化 -> 检索 -> 拼接上下文 -> 生成答案”串起来，是后面复现 RAG 的核心主线。


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

if section_ready("16.1", HAS_DASHSCOPE, "未找到百炼 API Key"):
    vectorstore = Chroma.from_texts(
        ["小明在华为工作", "熊喜欢吃蜂蜜", "LangChain 可以帮助开发 RAG 应用"],
        embedding=make_embeddings(),
        collection_name="section_16_retrieval",
        persist_directory=str(CHROMA_DIR / "section_16_retrieval"),
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
    prompt = ChatPromptTemplate.from_template(
        "只根据下面的上下文回答问题。\n上下文：{context}\n问题：{question}"
    )
    chain = (
        RunnableParallel(
            {
                "context": retriever,
                "question": RunnablePassthrough(),
            }
        )
        | prompt
        | make_chat_model(model_name="qwen-plus", temperature=0)
        | StrOutputParser()
    )
    print(chain.invoke("小明在哪里工作？"))


## 17-文档加载器

### 17.1 从本地 txt 目录批量加载文档

对应原笔记本里的文档加载基础。


In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

if section_ready("17.1", TEXT_DIR.exists(), "没有找到 txt 目录"):
    loader = DirectoryLoader(
        str(TEXT_DIR),
        glob="*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    docs = loader.load()
    print("doc count =", len(docs))
    for doc in docs[:2]:
        print("-" * 80)
        print(doc.metadata.get("source"))
        print(short_text(doc.page_content, 120))


## 18-处理知识库PDF文档

### 18.1 用 PyPDFLoader 读取 PDF

这一节先把 PDF 文本读出来，后面做 RAG 时会很常用。


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_candidates = sorted(PDF_DIR.glob("*.pdf")) if PDF_DIR.exists() else []
if section_ready("18.1", bool(pdf_candidates), "pdf 目录下没有找到 PDF 文件"):
    pdf_path = pdf_candidates[0]
    loader = PyPDFLoader(str(pdf_path))
    pdf_docs = loader.load()
    print("pdf =", pdf_path.name)
    print("pages =", len(pdf_docs))
    print(short_text(pdf_docs[0].page_content, 240))


## 19-HTML文本分割

### 19.1 HTMLHeaderTextSplitter

这一节对应 HTML 文档的结构化切分。


In [ ]:
from langchain_text_splitters import HTMLHeaderTextSplitter

html_path = TEXT_DIR / "test.html"
if section_ready("19.1", html_path.exists() and HAS_BS4, "缺少 HTML 文件或 beautifulsoup4"):
    html_text = html_path.read_text(encoding="utf-8", errors="ignore")
    splitter = HTMLHeaderTextSplitter(
        headers_to_split_on=[
            ("h1", "一级标题"),
            ("h2", "二级标题"),
        ]
    )
    docs = splitter.split_text(html_text)
    print("split count =", len(docs))
    if docs:
        print("metadata =", docs[0].metadata)
        print(short_text(docs[0].page_content, 240))


## 20-多种文本分割方式提炼内容

### 20.1 字符切分 / 递归切分 / Markdown 标题切分

这一节把常见切分方式放在一起比较。


In [ ]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

markdown_path = TEXT_DIR / "1.md"
if section_ready("20.1", markdown_path.exists(), "没有找到 txt/1.md"):
    markdown_text = markdown_path.read_text(encoding="utf-8", errors="ignore")

    char_splitter = CharacterTextSplitter(chunk_size=120, chunk_overlap=20)
    char_chunks = char_splitter.split_text(markdown_text)

    recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
    recursive_chunks = recursive_splitter.split_text(markdown_text)

    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[("#", "一级标题"), ("##", "二级标题")]
    )
    header_docs = header_splitter.split_text(markdown_text)

    print("character chunks =", len(char_chunks))
    print("recursive chunks =", len(recursive_chunks))
    print("markdown header docs =", len(header_docs))


## 21-多向量检索器

### 21.1 MultiVectorRetriever 最小示例

这一节演示“一个父文档对应多个子向量”的基本思想。


In [ ]:
import uuid

from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.documents import Document
from langchain_core.stores import InMemoryByteStore
from langchain_community.vectorstores import Chroma

if section_ready("21.1", HAS_DASHSCOPE, "未找到百炼 API Key"):
    vectorstore = Chroma(
        collection_name="section_21_multi_vector",
        embedding_function=make_embeddings(),
        persist_directory=str(CHROMA_DIR / "section_21_multi_vector"),
    )
    byte_store = InMemoryByteStore()
    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        byte_store=byte_store,
        id_key="doc_id",
    )

    parent_docs = [
        Document(page_content="LangChain 可以帮助我们把检索和生成串起来。", metadata={"source": "note-1"}),
        Document(page_content="向量检索适合语义相似的问答场景。", metadata={"source": "note-2"}),
    ]
    doc_ids = [str(uuid.uuid4()) for _ in parent_docs]
    child_docs = []

    for doc_id, doc in zip(doc_ids, parent_docs):
        for sentence in doc.page_content.split("。"):
            sentence = sentence.strip()
            if sentence:
                child_docs.append(Document(page_content=sentence, metadata={"doc_id": doc_id}))

    retriever.vectorstore.add_documents(child_docs)
    retriever.docstore.mset(list(zip(doc_ids, parent_docs)))

    docs = retriever.invoke("谁能把检索和生成串起来？")
    for doc in docs:
        print(doc.metadata, "=>", doc.page_content)


## 22-自查询

### 22.1 SelfQueryRetriever

这节保留“自然语言问题 -> 自动生成结构化筛选条件”的核心思路。


In [ ]:
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma

if section_ready("22.1", HAS_DASHSCOPE and HAS_LARK, "缺少百炼 API Key 或 lark"):
    docs = [
        Document(page_content="这是一门入门 LangChain 课程，适合新手。", metadata={"year": 2024, "level": "beginner"}),
        Document(page_content="这篇笔记讲多向量检索，适合有基础的同学。", metadata={"year": 2025, "level": "advanced"}),
        Document(page_content="这份资料介绍提示词模板。", metadata={"year": 2023, "level": "beginner"}),
    ]
    vectorstore = Chroma.from_documents(
        docs,
        embedding=make_embeddings(),
        collection_name="section_22_self_query",
        persist_directory=str(CHROMA_DIR / "section_22_self_query"),
    )
    metadata_field_info = [
        AttributeInfo(name="year", description="资料年份", type="integer"),
        AttributeInfo(name="level", description="难度等级", type="string"),
    ]
    retriever = SelfQueryRetriever.from_llm(
        make_chat_model(model_name="qwen-plus", temperature=0),
        vectorstore,
        "LangChain 学习资料",
        metadata_field_info,
        verbose=False,
        search_kwargs={"k": 3},
    )
    results = retriever.invoke("找 year 大于等于 2025 且 level 是 advanced 的资料")
    for doc in results:
        print(doc.metadata, "=>", doc.page_content)


## 23-大模型并行发散流程控制

### 23.1 RunnableParallel

对同一个主题并行产出“总结”和“追问问题”。


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel

if section_ready("23.1", HAS_DASHSCOPE, "未找到百炼 API Key"):
    parser = StrOutputParser()
    parallel_chain = RunnableParallel(
        {
            "summary": ChatPromptTemplate.from_template(
                "请用一句话总结 {topic}，不要使用表情符号。"
            )
            | make_chat_model(model_name="qwen-plus", temperature=0)
            | parser,
            "question": ChatPromptTemplate.from_template(
                "请给 {topic} 提一个追问问题，不要使用表情符号。"
            )
            | make_chat_model(model_name="qwen-plus", temperature=0)
            | parser,
        }
    )
    print(parallel_chain.invoke({"topic": "LangChain"}))


## 24-大模型路由映射传递数据

### 24.1 RunnablePassthrough + RunnableLambda 路由

这节展示如何根据输入内容把请求路由到不同处理链。


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

if section_ready("24.1", HAS_DASHSCOPE, "未找到百炼 API Key"):
    parser = StrOutputParser()
    router = RunnableLambda(lambda x: "rag" if "检索" in x["topic"] else "prompt")
    routes = {
        "rag": ChatPromptTemplate.from_template(
            "请解释 RAG 和 {topic} 的关系，不要使用表情符号。"
        )
        | make_chat_model(model_name="qwen-plus", temperature=0)
        | parser,
        "prompt": ChatPromptTemplate.from_template(
            "请解释提示词工程和 {topic} 的关系，不要使用表情符号。"
        )
        | make_chat_model(model_name="qwen-plus", temperature=0)
        | parser,
    }
    chain = RunnablePassthrough.assign(route=router) | RunnableLambda(
        lambda x: routes[x["route"]].invoke({"topic": x["topic"]})
    )
    print(chain.invoke({"topic": "检索增强生成"}))
    print("-" * 80)
    print(chain.invoke({"topic": "提示词模板"}))


## 25-设置配置项以适应运行需求

### 25.1 运行时配置一个 Runnable 的行为

这节用 `RunnableLambda` 读取运行时 `config`，演示“同一条链根据配置改变输出风格”。


In [ ]:
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.config import RunnableConfig

def explain_with_config(topic: str, config: RunnableConfig) -> str:
    metadata = config.get("metadata", {}) if config else {}
    style = metadata.get("style", "formal")
    if style == "casual":
        return f"轻松版：{topic} 就像给大模型搭乐高。"
    return f"正式版：{topic} 是一套用于组织大模型应用流程的框架。"

runner = RunnableLambda(explain_with_config)
print(runner.invoke("LangChain"))
print(runner.invoke("LangChain", config={"metadata": {"style": "casual"}}))


## 26-自定义流生成器函数

### 26.1 RunnableGenerator

这一节保留“自定义流式生成器”的核心概念。


In [ ]:
from typing import Iterator

from langchain_core.runnables import RunnableGenerator

def character_transform(chunks: Iterator[str]) -> Iterator[str]:
    for chunk in chunks:
        yield chunk.upper()

runner = RunnableGenerator(character_transform)
print("".join(runner.stream("langchain")))


## 27-添加消息历史记录（内存）

### 27.1 RunnableWithMessageHistory

这节把“消息历史记忆”串起来，也是后面做聊天型智能体时非常常见的模式。


In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

if section_ready("27.1", HAS_DASHSCOPE, "未找到百炼 API Key"):
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "你是一位会记住上下文的学习助理，回答使用纯文本。"),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{input}"),
        ]
    )
    chain = prompt | make_chat_model(model_name="qwen-plus", temperature=0)

    store: dict[str, BaseChatMessageHistory] = {}

    def get_session_history(session_id: str) -> BaseChatMessageHistory:
        if session_id not in store:
            store[session_id] = ChatMessageHistory()
        return store[session_id]

    with_history = RunnableWithMessageHistory(
        chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history",
    )

    first = with_history.invoke(
        {"input": "我最近在学 LangChain。"},
        config={"configurable": {"session_id": "demo-session"}},
    )
    second = with_history.invoke(
        {"input": "你还记得我在学什么吗？"},
        config={"configurable": {"session_id": "demo-session"}},
    )

    print(first.content)
    print("-" * 80)
    print(second.content)
